In [1]:
# Cell 1: 필수 라이브러리 설치
%pip install imagehash pillow requests --quiet


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2: 라이브러리 import 및 설정
import os
import json
import imagehash
from PIL import Image
import requests
from io import BytesIO
from difflib import SequenceMatcher
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIError, APIConnectionError, AuthenticationError

# 설정
SIMILARITY_THRESHOLD = 0.5  # 유사도 임계값 (0.5 이상인 것만 Vision API로 비교)
IMAGE_WEIGHT = 0.4  # 이미지 유사도 가중치
PRICE_WEIGHT = 0.2  # 가격 유사도 가중치
TITLE_WEIGHT = 0.4  # 상품명 유사도 가중치
MAX_WORKERS = 10  # 병렬 처리 스레드 수
TOTAL_WEIGHT = IMAGE_WEIGHT + PRICE_WEIGHT + TITLE_WEIGHT

VISION_MODEL = "gpt-4o-mini"
VISION_DETAIL = "low"
VISION_TOP_K = 3
VISION_MAX_RETRIES = 3
VISION_BASE_WAIT = 2
VISION_RATE_LIMIT_WAIT = 10

print("✅ 라이브러리 import 완료")
print(f"📊 유사도 임계값: {SIMILARITY_THRESHOLD}")
print(f"⚖️ 가중치 - 이미지: {IMAGE_WEIGHT}, 가격: {PRICE_WEIGHT}, 상품명: {TITLE_WEIGHT}")
print(f"⚡ 병렬 처리: 최대 {MAX_WORKERS}개 스레드")
print(f"👁️ Vision 모델: {VISION_MODEL} (detail={VISION_DETAIL}, TOP_K={VISION_TOP_K})\n")


✅ 라이브러리 import 완료
📊 유사도 임계값: 0.5
⚖️ 가중치 - 이미지: 0.4, 가격: 0.2, 상품명: 0.4
⚡ 병렬 처리: 최대 10개 스레드
👁️ Vision 모델: gpt-4o-mini (detail=low, TOP_K=3)



In [3]:
# Cell 3: 이미지 해싱 함수 (캐싱 포함)

# 전역 캐시 딕셔너리
_image_hash_cache = {}

def get_image_hash(image_url, timeout=3):
    """
    이미지 URL에서 perceptual hash를 계산합니다 (캐싱 포함).
    
    Args:
        image_url: 이미지 URL
        timeout: 요청 타임아웃 (초)
    
    Returns:
        imagehash.ImageHash 또는 None (실패 시)
    """
    # 캐시 확인
    if image_url in _image_hash_cache:
        return _image_hash_cache[image_url]
    
    try:
        response = requests.get(image_url, timeout=timeout)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))
        hash_value = imagehash.phash(img)  # perceptual hash 사용
        
        # 캐시에 저장
        _image_hash_cache[image_url] = hash_value
        return hash_value
    except Exception:
        # 실패한 경우 캐시에 저장하지 않아 일시적 오류 시 재시도 가능
        return None


def calculate_image_similarity(url1, url2):
    """
    두 이미지의 유사도를 계산합니다 (0-1, 1에 가까울수록 유사).
    
    Args:
        url1: 첫 번째 이미지 URL
        url2: 두 번째 이미지 URL
    
    Returns:
        float: 유사도 (0.0-1.0)
    """
    hash1 = get_image_hash(url1)
    hash2 = get_image_hash(url2)
    
    if hash1 is None or hash2 is None:
        return 0.0
    
    # Hamming distance 계산 (0-64)
    difference = hash1 - hash2
    
    # 유사도로 변환 (0-1)
    similarity = 1 - (difference / 64.0)
    return max(0.0, min(1.0, similarity))  # 0-1 범위로 제한


print("✅ 이미지 해싱 함수 정의 완료")
print("   - get_image_hash(): 이미지 URL에서 해시 계산")
print("   - calculate_image_similarity(): 두 이미지의 유사도 계산\n")


✅ 이미지 해싱 함수 정의 완료
   - get_image_hash(): 이미지 URL에서 해시 계산
   - calculate_image_similarity(): 두 이미지의 유사도 계산



In [4]:
# Cell 4: 상품명 유사도 계산 함수

def calculate_title_similarity(title1, title2):
    """
    두 상품명의 유사도를 계산합니다 (0-1, 1에 가까울수록 유사).
    
    Args:
        title1: 첫 번째 상품명
        title2: 두 번째 상품명
    
    Returns:
        float: 유사도 (0.0-1.0)
    """
    if not title1 or not title2:
        return 0.0
    
    # 소문자로 변환하여 비교
    title1_lower = title1.lower().strip()
    title2_lower = title2.lower().strip()
    
    # SequenceMatcher를 사용한 유사도 계산
    similarity = SequenceMatcher(None, title1_lower, title2_lower).ratio()
    
    return similarity


print("✅ 상품명 유사도 계산 함수 정의 완료")
print("   - calculate_title_similarity(): 두 상품명의 유사도 계산\n")


✅ 상품명 유사도 계산 함수 정의 완료
   - calculate_title_similarity(): 두 상품명의 유사도 계산



In [5]:
# Cell 5: 가격 유사도 계산 함수

def parse_price(price_value):
    """문자열/숫자 형태의 가격을 float로 변환합니다."""
    if price_value is None:
        return None
    if isinstance(price_value, (int, float)):
        return float(price_value)
    cleaned = re.sub(r"[^0-9.]", "", str(price_value))
    if not cleaned:
        return None
    try:
        return float(cleaned)
    except ValueError:
        return None


def calculate_price_similarity(price1, price2):
    """두 가격의 유사도(0-1)를 계산합니다."""
    p1 = parse_price(price1)
    p2 = parse_price(price2)

    if p1 is None or p2 is None or max(p1, p2) == 0:
        return 0.0

    diff_ratio = abs(p1 - p2) / max(p1, p2)
    return max(0.0, 1 - diff_ratio)


print("✅ 가격 유사도 계산 함수 정의 완료")
print("   - parse_price(): 가격 문자열을 float로 변환")
print("   - calculate_price_similarity(): 두 가격의 유사도 계산\n")


✅ 가격 유사도 계산 함수 정의 완료
   - parse_price(): 가격 문자열을 float로 변환
   - calculate_price_similarity(): 두 가격의 유사도 계산



In [6]:
# Cell 5: 종합 유사도 계산 및 필터링 함수

def calculate_combined_similarity(product1, product2):
    """
    이미지, 가격, 상품명 유사도를 종합하여 계산합니다.
    
    Args:
        product1: 첫 번째 상품 정보 (dict)
        product2: 두 번째 상품 정보 (dict)
    
    Returns:
        dict: {
            "combined_similarity": 종합 유사도 (0-1),
            "image_similarity": 이미지 유사도 (0-1),
            "price_similarity": 가격 유사도 (0-1),
            "title_similarity": 상품명 유사도 (0-1)
        }
    """
    # 이미지 유사도 계산
    img_url1 = product1.get("thumbnail_url", "")
    img_url2 = product2.get("thumbnail_url", "")
    image_sim = calculate_image_similarity(img_url1, img_url2) if img_url1 and img_url2 else 0.0
    
    # 상품명 유사도 계산
    title1 = product1.get("title", "")
    title2 = product2.get("title", "")
    title_sim = calculate_title_similarity(title1, title2)
    
    # 가격 유사도 계산 (표시가 우선, 없으면 원가)
    price1 = product1.get("displayed_price") or product1.get("price") or product1.get("original_price")
    price2 = product2.get("displayed_price") or product2.get("price") or product2.get("original_price")
    price_sim = calculate_price_similarity(price1, price2)
    
    # 가중 평균으로 종합 유사도 계산
    weighted_sum = (image_sim * IMAGE_WEIGHT) + (price_sim * PRICE_WEIGHT) + (title_sim * TITLE_WEIGHT)
    combined = weighted_sum / TOTAL_WEIGHT if TOTAL_WEIGHT else 0.0
    
    return {
        "combined_similarity": combined,
        "image_similarity": image_sim,
        "price_similarity": price_sim,
        "title_similarity": title_sim
    }


def filter_similar_products(products_coupang, products_ssadagu, threshold=SIMILARITY_THRESHOLD, max_workers=MAX_WORKERS):
    """
    쿠팡과 싸다구 상품 중 유사한 상품 쌍을 필터링합니다 (병렬 처리).
    
    Args:
        products_coupang: 쿠팡 상품 리스트
        products_ssadagu: 싸다구 상품 리스트
        threshold: 유사도 임계값 (기본값: SIMILARITY_THRESHOLD)
        max_workers: 병렬 처리 스레드 수 (기본값: MAX_WORKERS)
    
    Returns:
        list: 유사한 상품 쌍 리스트
    """
    candidates = []
    
    # 비교 작업 리스트 생성 (썸네일이 있는 것만)
    comparison_tasks = []
    for p1 in products_coupang:
        for p2 in products_ssadagu:
            if p1.get("thumbnail_url") and p2.get("thumbnail_url"):
                comparison_tasks.append((p1, p2))
    
    total_comparisons = len(comparison_tasks)
    total_possible = len(products_coupang) * len(products_ssadagu)
    
    print(f"🔍 총 {len(products_coupang)}개(쿠팡) × {len(products_ssadagu)}개(싸다구) = {total_possible}개 비교")
    print(f"   (썸네일 있는 상품: {total_comparisons}개 비교)")
    print(f"⚡ 병렬 처리: 최대 {max_workers}개 스레드 사용\n")
    
    completed = 0
    start_time = time.time()
    
    # 병렬 처리로 비교 수행
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 각 비교 작업을 제출
        future_to_task = {
            executor.submit(calculate_combined_similarity, p1, p2): (p1, p2)
            for p1, p2 in comparison_tasks
        }
        
        # 완료된 작업 처리
        for future in as_completed(future_to_task):
            completed += 1
            p1, p2 = future_to_task[future]
            
            # 진행 상황 출력 (10%마다 또는 완료 시)
            if completed % max(1, total_comparisons // 10) == 0 or completed == total_comparisons:
                progress = (completed / total_comparisons) * 100
                elapsed = time.time() - start_time
                if completed > 0:
                    estimated_total = elapsed / (completed / total_comparisons)
                    remaining = max(0, estimated_total - elapsed)
                    print(f"  진행률: {progress:.1f}% ({completed}/{total_comparisons}) | "
                          f"경과: {elapsed/60:.1f}분 | 예상 남은 시간: {remaining/60:.1f}분")
            
            try:
                similarity = future.result()
                
                # 임계값 이상인 경우만 후보에 추가
                if similarity["combined_similarity"] >= threshold:
                    candidates.append({
                        "coupang_product": {
                            "title": p1.get("title", ""),
                            "price": p1.get("displayed_price") or p1.get("price") or p1.get("original_price", ""),
                            "thumbnail_url": p1.get("thumbnail_url", ""),
                            "product_link": p1.get("product_link", "")
                        },
                        "ssadagu_product": {
                            "title": p2.get("title", ""),
                            "price": p2.get("displayed_price") or p2.get("price") or p2.get("original_price", ""),
                            "thumbnail_url": p2.get("thumbnail_url", ""),
                            "product_link": p2.get("product_link", "")
                        },
                        "similarity": similarity
                    })
            except Exception as e:
                print(f"  ⚠️ 비교 실패: {str(e)[:50]}")
    
    elapsed_total = time.time() - start_time
    print(f"\n✅ 필터링 완료: {len(candidates)}개 후보 발견 (임계값: {threshold} 이상)")
    print(f"⏱️ 총 소요 시간: {elapsed_total/60:.1f}분")
    if elapsed_total > 0:
        print(f"📊 평균 처리 속도: {total_comparisons/(elapsed_total/60):.1f}개 비교/분\n")
    return candidates


print("✅ 종합 유사도 계산 및 필터링 함수 정의 완료")
print("   - calculate_combined_similarity(): 이미지 + 상품명 종합 유사도")
print("   - filter_similar_products(): 유사한 상품 쌍 필터링\n")


✅ 종합 유사도 계산 및 필터링 함수 정의 완료
   - calculate_combined_similarity(): 이미지 + 상품명 종합 유사도
   - filter_similar_products(): 유사한 상품 쌍 필터링



In [7]:
# Cell 6: 비교 및 Vision 실행 함수 정의

def parse_vision_response(raw_text: str):
    """```json 코드 블록을 포함한 Vision 응답을 안전하게 파싱합니다."""
    text = (raw_text or "").strip()
    if not text:
        return None
    code_block = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if code_block:
        text = code_block.group(1).strip()
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            parsed.setdefault("status", "ok")
        return parsed
    except json.JSONDecodeError:
        return None


def call_vision_api_with_retry(
    client,
    content,
    max_retries=VISION_MAX_RETRIES,
    base_wait=VISION_BASE_WAIT,
    rate_limit_wait=VISION_RATE_LIMIT_WAIT,
):
    """Vision API 호출을 재시도 로직과 함께 수행합니다."""
    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=VISION_MODEL,
                messages=content,
                max_tokens=400,
                timeout=40,
            )
            return response, None
        except AuthenticationError as e:
            return None, f"재시도 불가 인증 오류: {type(e).__name__}: {e}"
        except RateLimitError as e:
            wait_time = rate_limit_wait * (attempt + 1)
            print(f"  ⚠️ Rate limit 발생. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
            last_error = f"Rate limit: {e}"
        except APIConnectionError as e:
            wait_time = base_wait ** (attempt + 1)
            print(f"  ⚠️ 네트워크 오류. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
            last_error = f"Connection error: {e}"
        except APIError as e:
            status = getattr(e, "status_code", None)
            if status and 400 <= status < 500:
                return None, f"재시도 불가 API 오류 (status {status}): {e}"
            wait_time = base_wait ** (attempt + 1)
            print(f"  ⚠️ API 오류. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
            last_error = f"API error: {e}"
        except Exception as e:
            if attempt == max_retries - 1:
                last_error = f"Unexpected error: {type(e).__name__}: {e}"
                break
            wait_time = base_wait ** (attempt + 1)
            print(f"  ⚠️ 일시적 오류. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
            last_error = str(e)
    return None, last_error or "Vision API 호출 실패"


def analyze_pair_with_vision(client, candidate, detail=VISION_DETAIL):
    """썸네일 2장을 Vision API로 비교하고 결과와 원문을 반환합니다."""
    coupang = candidate.get("coupang_product", {})
    ssadagu = candidate.get("ssadagu_product", {})
    coupang_thumb = coupang.get("thumbnail_url", "")
    ssadagu_thumb = ssadagu.get("thumbnail_url", "")

    if not coupang_thumb or not ssadagu_thumb:
        return {
            "status": "skipped",
            "reason": "missing_thumbnail",
            "message": "썸네일 URL이 없어 Vision 비교를 건너뜁니다.",
        }, None

    prompt = f"""두 쇼핑몰 썸네일이 같은 제품을 나타내는지 비교하세요. 반드시 JSON으로만 답변하세요.
출력 형식:
{{
  \"isSameProduct\": \"y\" 또는 \"n\",
  \"confidence\": 0-100 사이 숫자,
  \"keySimilarities\": ["항목"],
  \"keyDifferences\": ["항목"],
  \"verdict\": "간단한 판단 이유"
}}
제품 정보:
- Coupang: {coupang.get('title', 'N/A')} / 가격 {coupang.get('price', 'N/A')}
- Ssadagu: {ssadagu.get('title', 'N/A')} / 가격 {ssadagu.get('price', 'N/A')}"""

    content = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": coupang_thumb, "detail": detail}},
                {"type": "image_url", "image_url": {"url": ssadagu_thumb, "detail": detail}},
            ],
        }
    ]

    response, error_msg = call_vision_api_with_retry(client, content)
    if response is None:
        return {"status": "request_failed", "message": error_msg}, None

    raw_text = response.choices[0].message.content.strip()
    parsed = parse_vision_response(raw_text)
    if parsed is None:
        return {"status": "parse_error", "raw_response": raw_text}, raw_text
    return parsed, raw_text


def find_project_root(start_dir: Path, max_depth: int = 5) -> Path:
    """현재 경로에서 상위 디렉토리를 탐색하며 프로젝트 루트를 추정합니다."""
    current = start_dir
    for _ in range(max_depth):
        if (current / ".git").exists() or (current / "pyproject.toml").exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    return start_dir


def run_compare_coupang_ssadagu(
    coupang_json_path,
    ssadagu_json_path,
    compare_output_path,
    vision_output_path,
    run_vision=True,
    top_k=VISION_TOP_K,
    vision_detail=VISION_DETAIL,
    max_workers=MAX_WORKERS,
):
    """쿠팡-싸다구 비교 파이프라인을 실행하고 결과를 저장합니다."""
    coupang_path = Path(coupang_json_path)
    ssadagu_path = Path(ssadagu_json_path)
    if not coupang_path.exists():
        raise FileNotFoundError(f"쿠팡 데이터 파일을 찾을 수 없습니다: {coupang_path}")
    if not ssadagu_path.exists():
        raise FileNotFoundError(f"싸다구 데이터 파일을 찾을 수 없습니다: {ssadagu_path}")

    print("📂 JSON 파일 로드 중...\n")
    try:
        with open(coupang_path, "r", encoding="utf-8") as f:
            coupang_data = json.load(f)
        with open(ssadagu_path, "r", encoding="utf-8") as f:
            ssadagu_data = json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON 파싱 오류: {e}")

    coupang_products = coupang_data.get("products", [])
    ssadagu_products = ssadagu_data.get("products", [])

    print(f"✅ 쿠팡 상품: {len(coupang_products)}개")
    print(f"✅ 싸다구 상품: {len(ssadagu_products)}개")
    print(f"📊 검색 키워드: {coupang_data.get('search_keyword', 'N/A')}\n")

    print("=" * 60)
    similar_candidates = filter_similar_products(
        coupang_products,
        ssadagu_products,
        threshold=SIMILARITY_THRESHOLD,
        max_workers=max_workers,
    )
    print("=" * 60)

    compare_output = Path(compare_output_path)
    compare_output.parent.mkdir(parents=True, exist_ok=True)
    result_payload = {
        "search_keyword": coupang_data.get("search_keyword", ""),
        "comparison_date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "settings": {
            "similarity_threshold": SIMILARITY_THRESHOLD,
            "image_weight": IMAGE_WEIGHT,
            "price_weight": PRICE_WEIGHT,
            "title_weight": TITLE_WEIGHT,
        },
        "statistics": {
            "total_coupang_products": len(coupang_products),
            "total_ssadagu_products": len(ssadagu_products),
            "total_comparisons": len(coupang_products) * len(ssadagu_products),
            "candidates_found": len(similar_candidates),
        },
        "candidates": similar_candidates,
    }

    with open(compare_output, "w", encoding="utf-8") as f:
        json.dump(result_payload, f, ensure_ascii=False, indent=2)

    print(f"✅ 결과 저장 완료: {compare_output}")
    print("📊 비교 결과 요약:")
    print(f"   - 쿠팡 상품 수: {len(coupang_products)}개")
    print(f"   - 싸다구 상품 수: {len(ssadagu_products)}개")
    print(f"   - 총 비교 횟수: {len(coupang_products) * len(ssadagu_products)}회")
    print(f"   - 유사한 상품 쌍: {len(similar_candidates)}개 (임계값: {SIMILARITY_THRESHOLD})\n")

    vision_results = None
    if not run_vision:
        print("⚠️ Vision 비교가 비활성화되어 있어 이미지 정밀 비교를 건너뜁니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    if not similar_candidates:
        print("⚠️ Vision 비교를 수행할 후보가 없습니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    project_root = find_project_root(Path.cwd())
    env_path = project_root / ".env"
    if not env_path.exists():
        print(f"⚠️ .env 파일을 찾을 수 없어 Vision 비교를 건너뜁니다: {env_path}")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    load_dotenv(env_path)
    api_key = os.getenv("OPENAI_API_KEY", "").strip()
    if not api_key:
        print("⚠️ OPENAI_API_KEY가 설정되어 있지 않아 Vision 비교를 건너뜁니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }
    if not api_key.startswith("sk-"):
        print("⚠️ OPENAI_API_KEY 형식이 올바르지 않아 Vision 비교를 건너뜁니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    client = OpenAI(api_key=api_key)
    top_candidates = sorted(
        similar_candidates,
        key=lambda x: x["similarity"]["combined_similarity"],
        reverse=True,
    )[:max(1, top_k)]

    if not top_candidates:
        print("⚠️ Vision 비교를 수행할 후보가 없습니다.")
        return {
            "coupang_products": coupang_products,
            "ssadagu_products": ssadagu_products,
            "similar_candidates": similar_candidates,
            "vision_results": None,
        }

    print(f"🔎 Vision API 정밀 비교 시작 (상위 {len(top_candidates)}개 후보, detail='{vision_detail}')\n")
    vision_results = []
    for idx, candidate in enumerate(top_candidates, 1):
        coupang = candidate["coupang_product"]
        ssadagu = candidate["ssadagu_product"]
        combined = candidate["similarity"]["combined_similarity"]
        print("=" * 70)
        print(f"[{idx}] 종합 유사도 {combined:.2%}")
        print(f"  Coupang : {coupang.get('title', '')[:80]}...")
        print(f"             가격 {coupang.get('price')}")
        print(f"  Ssadagu : {ssadagu.get('title', '')[:80]}...")
        print(f"             가격 {ssadagu.get('price')}")

        result, raw_text = analyze_pair_with_vision(client, candidate, detail=vision_detail)
        vision_results.append({
            "candidate": candidate,
            "vision_result": result,
            "raw_text": raw_text,
        })

        status = result.get("status")
        if status == "skipped":
            print(f"  ⚠️ Vision 비교 스킵: {result.get('message')}")
        elif status == "request_failed":
            print(f"  ⚠️ Vision API 호출 실패: {result.get('message')}")
        elif status == "parse_error":
            preview = (result.get("raw_response") or "")[:200]
            print("  ⚠️ JSON 파싱 실패. 원문 미리보기:")
            print(f"     {preview}...")
        else:
            print("  ✅ Vision 결과:")
            print(f"     동일 여부 : {result.get('isSameProduct', 'N/A')}, 신뢰도 {result.get('confidence', 'N/A')}%")
            similarities = result.get("keySimilarities") or []
            if similarities:
                print(f"     공통점   : {', '.join(similarities[:3])}")
            differences = result.get("keyDifferences") or []
            if differences:
                print(f"     차이점   : {', '.join(differences[:3])}")
            if result.get("verdict"):
                print(f"     판단 사유: {result['verdict']}")

    vision_output = Path(vision_output_path)
    vision_output.parent.mkdir(parents=True, exist_ok=True)
    export_payload = {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "model": VISION_MODEL,
        "detail": vision_detail,
        "top_k": len(top_candidates),
        "results": [
            {
                "coupang_product": item["candidate"]["coupang_product"],
                "ssadagu_product": item["candidate"]["ssadagu_product"],
                "similarity": item["candidate"]["similarity"],
                "vision_result": item["vision_result"],
            }
            for item in vision_results
        ],
    }
    with open(vision_output, "w", encoding="utf-8") as f:
        json.dump(export_payload, f, ensure_ascii=False, indent=2)

    print("\n🎯 Vision 비교 완료. 결과가 저장되었습니다.")
    print(f"📝 저장 경로: {vision_output}")

    return {
        "coupang_products": coupang_products,
        "ssadagu_products": ssadagu_products,
        "similar_candidates": similar_candidates,
        "vision_results": vision_results,
    }


In [8]:
# Cell 7: 실행 스크립트 생성 및 실행

import os
import sys
import tempfile
import subprocess
import inspect
import textwrap

COUPANG_JSON_PATH = Path("../crawling_tests/coupang_search_results.json").resolve()
SSADAGU_JSON_PATH = Path("../crawling_tests/ssadagu_search_results.json").resolve()
COMPARE_OUTPUT_JSON = Path("compare_coupang_ssadagu_results.json").resolve()
VISION_OUTPUT_JSON = Path("vision_top3_results.json").resolve()
RUN_VISION = True
VISION_TOP_K_OVERRIDE = VISION_TOP_K
VISION_DETAIL_LEVEL = VISION_DETAIL
MAX_WORKERS_OVERRIDE = MAX_WORKERS

print("▶ 쿠팡 vs 싸다구 상품 비교를 시작합니다...\n")

function_list = [
    get_image_hash,
    calculate_image_similarity,
    calculate_title_similarity,
    parse_price,
    calculate_price_similarity,
    calculate_combined_similarity,
    filter_similar_products,
    parse_vision_response,
    call_vision_api_with_retry,
    analyze_pair_with_vision,
    find_project_root,
    run_compare_coupang_ssadagu,
]

function_sources = [textwrap.dedent(inspect.getsource(fn)) for fn in function_list]
functions_code = "\n\n".join(function_sources)

coupang_literal = repr(str(COUPANG_JSON_PATH))
ssadagu_literal = repr(str(SSADAGU_JSON_PATH))
compare_literal = repr(str(COMPARE_OUTPUT_JSON))
vision_literal = repr(str(VISION_OUTPUT_JSON))

script_lines = [
    "import os",
    "import json",
    "import time",
    "import re",
    "from pathlib import Path",
    "from io import BytesIO",
    "from concurrent.futures import ThreadPoolExecutor, as_completed",
    "from difflib import SequenceMatcher",
    "import requests",
    "import imagehash",
    "from PIL import Image",
    "from dotenv import load_dotenv",
    "from openai import OpenAI, RateLimitError, APIError, APIConnectionError, AuthenticationError",
    "",
    f"SIMILARITY_THRESHOLD = {SIMILARITY_THRESHOLD}",
    f"IMAGE_WEIGHT = {IMAGE_WEIGHT}",
    f"PRICE_WEIGHT = {PRICE_WEIGHT}",
    f"TITLE_WEIGHT = {TITLE_WEIGHT}",
    f"MAX_WORKERS = {MAX_WORKERS_OVERRIDE}",
    "TOTAL_WEIGHT = IMAGE_WEIGHT + PRICE_WEIGHT + TITLE_WEIGHT",
    "",
    "_image_hash_cache = {}",
    "",
    f"VISION_MODEL = {VISION_MODEL!r}",
    f"VISION_DETAIL = {VISION_DETAIL_LEVEL!r}",
    f"VISION_TOP_K = {VISION_TOP_K_OVERRIDE}",
    f"VISION_MAX_RETRIES = {VISION_MAX_RETRIES}",
    f"VISION_BASE_WAIT = {VISION_BASE_WAIT}",
    f"VISION_RATE_LIMIT_WAIT = {VISION_RATE_LIMIT_WAIT}",
    "",
    functions_code,
    "",
    'if __name__ == "__main__":',
    f"    run_compare_coupang_ssadagu(",
    f"        coupang_json_path={coupang_literal},",
    f"        ssadagu_json_path={ssadagu_literal},",
    f"        compare_output_path={compare_literal},",
    f"        vision_output_path={vision_literal},",
    f"        run_vision={RUN_VISION},",
    f"        top_k={VISION_TOP_K_OVERRIDE},",
    f"        vision_detail={VISION_DETAIL_LEVEL!r},",
    f"        max_workers={MAX_WORKERS_OVERRIDE},",
    "    )",
    '    print("\\n✅ 비교 스크립트 실행 완료")',
]

script_content = "\n".join(script_lines)

fd, script_path = tempfile.mkstemp(suffix="_compare_coupang_ssadagu.py", text=True)
os.close(fd)
with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_content)

try:
    completed = subprocess.run(
        [sys.executable, script_path],
        capture_output=True,
        text=True,
        check=False,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
finally:
    try:
        os.remove(script_path)
    except OSError:
        pass


▶ 쿠팡 vs 싸다구 상품 비교를 시작합니다...

📂 JSON 파일 로드 중...

✅ 쿠팡 상품: 30개
✅ 싸다구 상품: 30개
📊 검색 키워드: 컴퓨터

🔍 총 30개(쿠팡) × 30개(싸다구) = 900개 비교
   (썸네일 있는 상품: 900개 비교)
⚡ 병렬 처리: 최대 10개 스레드 사용

  진행률: 10.0% (90/900) | 경과: 0.9분 | 예상 남은 시간: 8.0분
  진행률: 20.0% (180/900) | 경과: 1.3분 | 예상 남은 시간: 5.1분
  진행률: 30.0% (270/900) | 경과: 1.7분 | 예상 남은 시간: 3.9분
  진행률: 40.0% (360/900) | 경과: 2.0분 | 예상 남은 시간: 3.0분
  진행률: 50.0% (450/900) | 경과: 2.3분 | 예상 남은 시간: 2.3분
  진행률: 60.0% (540/900) | 경과: 2.7분 | 예상 남은 시간: 1.8분
  진행률: 70.0% (630/900) | 경과: 3.0분 | 예상 남은 시간: 1.3분
  진행률: 80.0% (720/900) | 경과: 3.4분 | 예상 남은 시간: 0.8분
  진행률: 90.0% (810/900) | 경과: 3.7분 | 예상 남은 시간: 0.4분
  진행률: 100.0% (900/900) | 경과: 4.0분 | 예상 남은 시간: 0.0분

✅ 필터링 완료: 7개 후보 발견 (임계값: 0.5 이상)
⏱️ 총 소요 시간: 4.0분
📊 평균 처리 속도: 223.6개 비교/분

✅ 결과 저장 완료: C:\Users\주민우\Final-AI-Fork\dev\compare_test\compare_coupang_ssadagu_results.json
📊 비교 결과 요약:
   - 쿠팡 상품 수: 30개
   - 싸다구 상품 수: 30개
   - 총 비교 횟수: 900회
   - 유사한 상품 쌍: 7개 (임계값: 0.5)

🔎 Vision API 정밀 비교 시작 (상위 3개 후보, detail='low')

[1] 